# Pipeline: Qualidade do Ar × Clima nas Capitais
Executa o pipeline ponta a ponta: **extração → silver → gold → data quality → gravação**.
Para rodar: configure o `key.env` (veja o README) e clique em **Run all**.

In [0]:
%pip install -r requirements.txt
# instala as bibliotecas listadas no requirements.txt (reprodutibilidade)

In [0]:
# ===== BLOCO: configuração do log =====
# logging: mostra no notebook as mensagens de cada etapa
import logging
# nível INFO: mostra mensagens de rotina, avisos e erros; force=True substitui configurações anteriores
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s", force=True)

# ===== BLOCO: importa os módulos do projeto =====
# configurações centrais
from src import config
# etapa de extração
from src.extract import extrair
# etapas de transformação
from src.transform import silver_medicoes, silver_clima, gold_qualidade_ar
# etapa de data quality
from src.quality import gerar_relatorio, salvar_relatorio
# etapa de gravação
from src.load import gravar_gold, copiar_para_output

# ===== BLOCO: garante que o schema e o Volume existem (idempotente) =====
# cria o schema do projeto, se não existir
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.desafio_ft03")
# cria o Volume onde ficam bronze, silver e gold, se não existir
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.desafio_ft03.dados")

## 1. Extração (bronze)

In [0]:
# baixa os dados do OpenAQ e da NASA POWER e grava o JSON bruto na bronze
resumo_extracao = extrair()
# mostra quantas estações/sensores deram certo e as falhas
print(resumo_extracao)

## 2. Transformação (silver e gold)

In [0]:
# silver: medições do OpenAQ desaninhadas, tipadas, limpas e deduplicadas
medicoes = silver_medicoes(spark)
# silver: clima da NASA desaninhado e pivotado
clima = silver_clima(spark)
# gold: cruzamento das duas fontes e enriquecimento
gold = gold_qualidade_ar(spark, medicoes, clima)
# guarda a gold em memória, porque ela será lida várias vezes (qualidade e gravação)
gold = gold.cache()
# mostra o volume da tabela final
print("linhas na gold:", gold.count())

## 3. Data quality

In [0]:
# executa as validações e monta o relatório
relatorio = gerar_relatorio(gold, resumo_extracao)
# salva o relatório no Volume (histórico da execução)
salvar_relatorio(relatorio, f"{config.VOLUME}/relatorios/relatorio_qualidade.json")
# salva uma cópia no output/ do repositório (entregável)
salvar_relatorio(relatorio, str(config.PASTA_OUTPUT / "relatorio_qualidade.json"))
# mostra o status geral e o de cada verificação
print("STATUS GERAL:", relatorio["status_geral"])
for nome, verificacao in relatorio["verificacoes"].items():
    print(f"  {nome}: {verificacao['status']}")

# portão de qualidade: se reprovou, interrompe antes de gravar dado defeituoso
if relatorio["status_geral"] == "reprovado":
    raise RuntimeError("Data quality reprovado: gravação cancelada. Veja relatorio_qualidade.json.")

## 4. Gravação (Parquet particionado)

In [0]:
# grava a gold em Parquet + Snappy, particionada por ano/mês
gravar_gold(gold)
# copia os .parquet para a pasta output/ do repositório
print("arquivos copiados para output/:", copiar_para_output())